# Exploring some ideas to filter
Goal: get rather non-official versions such as amateur covers.

Ideas:
- filter by stopwords (e.g. "official")
- filter by channels (e.g. )
- filter by ambiguity of entities (e.g. the artist "the the" did not work well when fuzzy matching)

In [33]:
import pandas as pd

df = pd.read_csv('data/matched/filtered_types.csv', sep="\t")

# Focus only on matches where title and artist matches
filtered = df.loc[df.match_type == 'both'].drop_duplicates(
    subset="youtube_id"
)


In [34]:
metadata = pd.read_json("data/metadata_filtered.jsonl", lines=True, orient='records')
metadata = metadata.loc[metadata.id.isin(filtered.youtube_id),:]

new_columns = ["clique_id"] + metadata.columns.tolist()
metadata = pd.merge(
    filtered.reset_index(),
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)
metadata = metadata[new_columns]


## Filter by stopwords

In [35]:
import re

stopwords = ["remaster", "official", "music video", "lyric video"]

pattern = "|".join(re.escape(word) for word in stopwords)

metadata_filtered = metadata[~metadata.title.str.contains(pattern, case=False, na=False)]

print(f"Removed {len(metadata) - len(metadata_filtered)} entries with stopwords in title")


Removed 56298 entries with stopwords in title


## By channel name
- remove where endswith `VEVO`
- remove where `lyric videos` is contained
- remove `- Topic` channels. See: https://support.google.com/youtube/answer/7636475?hl=en#zippy=%2Chow-does-youtube-decide-when-to-auto-generate-a-topic-channel-for-an-artist



In [36]:
metadata_filtered.loc[:,"channel_name"] = metadata_filtered.channel.apply(lambda x: x["name"])

# filtering masks
mask_vevo = metadata_filtered.channel_name.str.endswith("VEVO")
mask_lyric_videos = metadata_filtered.channel_name.str.contains("lyric video", case=False, na=False)
mask_topic = metadata_filtered.channel_name.str.endswith("- Topic")

metadata_filtered = metadata_filtered.loc[~mask_vevo & ~mask_lyric_videos & ~mask_topic, :]
print(f"Removed {mask_vevo.sum()} VEVO entries, {mask_lyric_videos.sum()} lyric video entries, and {mask_topic.sum()} Topic entries")
print(f"Resulting dataset has {len(metadata_filtered)} entries")


/tmp/ipykernel_4179926/2005902755.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered.loc[:,"channel_name"] = metadata_filtered.channel.apply(lambda x: x["name"])


Removed 2873 VEVO entries, 102 lyric video entries, and 385054 Topic entries
Resulting dataset has 992705 entries
